# 🌾 Rice Vision AI — Colab Inference Server

Notebook này vận hành dịch vụ suy luận Rice Vision AI trên Google Colab GPU (Tesla T4) với ngrok tunnel.

### Hướng dẫn vận hành nhanh:
1. **GPU Runtime**: Chọn menu **Runtime** → **Change runtime type** → **T4 GPU**.
2. **Mount Drive & Đường dẫn**: Chạy Cell 2 để mount Drive và trỏ `PROJECT_ROOT` tới thư mục `MAIN_SOURCES`.
3. **Cài đặt thư viện**: Chạy Cell 3 để cài đặt đúng gói theo `requirements-colab.txt`.
4. **Cấu hình & Model**: Khai báo token ngrok và đường dẫn model trong file `AI_SERVICES/.env`. Cell 4 nạp cấu hình tự động.
5. **Preflight**: Cell 5 nạp trước các model và kiểm tra tính hợp lệ trước khi khởi chạy mạng.
6. **Khởi chạy Server**: Cell 6 khởi chạy FastAPI server cục bộ, kiểm tra liveness và mở ngrok tunnel ra Internet.
7. **Dừng / Đổi model**: Khi cần đổi model, chạy Cell 8 để đóng tunnel và tắt server, sửa `.env` rồi chạy lại từ Cell 4.


In [1]:
# Cell 2: Mount Google Drive & Khai báo PROJECT_ROOT duy nhất
from google.colab import drive
import os
import sys
from pathlib import Path

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Khai báo PROJECT_ROOT tới MAIN_SOURCES (Chỉnh sửa nếu vị trí thư mục của bạn khác)
PROJECT_ROOT = Path('/content/drive/MyDrive/NGHIÊN CỨU KHOA HỌC/GROUP_MEMBERS/NGUYEN MINH TRI/MAIN_SOURCES')

# 3. Kiểm tra tính hợp lệ của cây thư mục
AI_SERVICES_DIR = PROJECT_ROOT / 'AI_SERVICES'
SRC_DIR = AI_SERVICES_DIR / 'src'
RICE_AI_DIR = SRC_DIR / 'rice_ai'
ENV_FILE = AI_SERVICES_DIR / '.env'

if not RICE_AI_DIR.is_dir():
    raise FileNotFoundError(
        f"❌ Không tìm thấy package mã nguồn tại: {RICE_AI_DIR}\n"
        f"Vui lòng kiểm tra lại giá trị PROJECT_ROOT trên Google Drive!"
    )

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
if str(AI_SERVICES_DIR) not in sys.path:
    sys.path.insert(0, str(AI_SERVICES_DIR))

print(f"✅ PROJECT_ROOT: {PROJECT_ROOT}")
print(f"✅ AI_SERVICES_DIR: {AI_SERVICES_DIR}")
print("✅ Source package 'rice_ai' đã sẵn sàng trong sys.path.")


Mounted at /content/drive
✅ PROJECT_ROOT: /content/drive/MyDrive/NGHIÊN CỨU KHOA HỌC/GROUP_MEMBERS/NGUYEN MINH TRI/MAIN_SOURCES
✅ AI_SERVICES_DIR: /content/drive/MyDrive/NGHIÊN CỨU KHOA HỌC/GROUP_MEMBERS/NGUYEN MINH TRI/MAIN_SOURCES/AI_SERVICES
✅ Source package 'rice_ai' đã sẵn sàng trong sys.path.


In [2]:
# Cell 3: Cài đặt Dependencies từ requirements-colab.txt & Kiểm tra phần cứng
import sys
import subprocess
import torch

# 1. Cài đặt các gói phụ thuộc
req_colab = AI_SERVICES_DIR / 'requirements-colab.txt'
if req_colab.is_file():
    print(f"📦 Đang cài đặt thư viện từ {req_colab}...")
    cmd = [sys.executable, "-m", "pip", "install", "-q", "-r", str(req_colab)]
    res = subprocess.run(cmd, capture_output=True, text=True)
    if res.returncode != 0:
        print(f"⚠️ Cảnh báo cài đặt: {res.stderr[:500]}")
    else:
        print("✅ Đã cài đặt hoàn tất các thư viện Colab.")
else:
    print(f"⚠️ Không tìm thấy {req_colab}. Tiếp tục với môi trường hiện có.")

# 2. Kiểm tra thông tin phần cứng và GPU
print("\n--- Môi trường tính toán ---")
print(f"Python: {sys.version.split()[0]}")
import sklearn
print(f"scikit-learn: {sklearn.__version__}")
import numpy as np
print(f"NumPy: {np.__version__}")
print(f"PyTorch: {torch.__version__}")

cuda_avail = torch.cuda.is_available()
print(f"CUDA Available: {cuda_avail}")
if cuda_avail:
    device_name = torch.cuda.get_device_name(0)
    print(f"GPU Device: {device_name}")
else:
    print("⚠️ CẢNH BÁO: Không có GPU! Vào menu Runtime -> Change runtime type -> T4 GPU để tăng tốc.")


📦 Đang cài đặt thư viện từ /content/drive/MyDrive/NGHIÊN CỨU KHOA HỌC/GROUP_MEMBERS/NGUYEN MINH TRI/MAIN_SOURCES/AI_SERVICES/requirements-colab.txt...
✅ Đã cài đặt hoàn tất các thư viện Colab.

--- Môi trường tính toán ---
Python: 3.13.15
scikit-learn: 1.6.1
NumPy: 2.1.3
PyTorch: 2.11.0+cu128
CUDA Available: True
GPU Device: Tesla T4


In [3]:
# Cell 4: Nạp Settings và cấu hình notebook từ .env
import os
from dotenv import dotenv_values
from rice_ai.settings import Settings

# Settings quản lý đường dẫn model, thiết bị YOLO, port và concurrency.
settings = Settings(env_file=ENV_FILE)

# Ngrok là cấu hình vận hành của notebook, không thuộc Settings của inference runtime.
file_env = dotenv_values(ENV_FILE) if ENV_FILE.is_file() else {}
def notebook_env_value(key: str, default: str = "") -> str:
    value = os.environ[key] if key in os.environ else file_env.get(key, default)
    return str(value or default).strip()

server_host = "127.0.0.1"
server_port = settings.port_ai
ngrok_auth_token = notebook_env_value("NGROK_AUTH_TOKEN")
ngrok_domain = notebook_env_value("NGROK_DOMAIN")

# Thông báo thiết lập cốt lõi; không in token bí mật.
print("--- Cấu hình Hệ thống (Settings) ---")
print(f"Host / Port      : {server_host} / {server_port}")
print(f"YOLO Device      : {settings.yolo_device_str} (Giải quyết: {settings.get_yolo_device()})")
print(f"YOLO Model Path  : {settings.get_yolo_path()}")
print(f"CNN Model Path   : {settings.get_cnn_path()}")
print(f"Regression Dir   : {settings.get_regression_dir()}")
print(f"Max Concurrency  : {settings.max_concurrent_inferences}")
print(f"Ngrok Auth Token : {'Đã cấu hình (Bảo mật)' if ngrok_auth_token else 'CHƯA CẤU HÌNH ⚠️'}")
print(f"Ngrok Domain     : {ngrok_domain or '(Sinh ngẫu nhiên)'}")


--- Cấu hình Hệ thống (Settings) ---
Host / Port      : 127.0.0.1 / 8000
YOLO Device      : auto (Giải quyết: cuda:0)
YOLO Model Path  : /content/drive/MyDrive/NGHIÊN CỨU KHOA HỌC/GROUP_MEMBERS/NGUYEN MINH TRI/MAIN_SOURCES/RESULTS/all-new-data-v1.yolov8_yolov8s-seg_trained/weights/best.pt
CNN Model Path   : /content/drive/MyDrive/NGHIÊN CỨU KHOA HỌC/GROUP_MEMBERS/NGUYEN MINH TRI/MAIN_SOURCES/RESULTS/CNN_DenseNet121_Trained/best_v3_step2.keras
Regression Dir   : /content/drive/MyDrive/NGHIÊN CỨU KHOA HỌC/GROUP_MEMBERS/NGUYEN MINH TRI/MAIN_SOURCES/LINEAR_REGRESSION_MODEL/models/ard
Max Concurrency  : 1
Ngrok Auth Token : Đã cấu hình (Bảo mật)
Ngrok Domain     : provolone-duress-probably.ngrok-free.dev


In [4]:
# Cell 5: Preflight Verification — Kiểm tra tính toàn vẹn của mô hình trước khi mở mạng
from rice_ai.models.vision_models import VisionModelProvider
from rice_ai.models.regression_loader import LoadedRegressionProvider

print("🔍 Đang tiến hành Preflight Verification...")

# 1. Khởi tạo và nạp kiểm tra Vision Models (YOLO & CNN)
vision_prov = VisionModelProvider(settings)
try:
    yolo_model = vision_prov.get_yolo_model()
    print(f"✅ YOLO Model đã nạp thành công trên thiết bị: {settings.get_yolo_device()}")
except Exception as e:
    raise RuntimeError(f"❌ Nạp YOLO Model thất bại: {e}")

try:
    cnn_model = vision_prov.get_cnn_model()
    print("✅ CNN Classifier (DenseNet121) đã nạp thành công.")
except Exception as e:
    raise RuntimeError(f"❌ Nạp CNN Model thất bại: {e}")

# 2. Khởi tạo và nạp kiểm tra Regression Bundle
reg_prov = LoadedRegressionProvider(settings)
try:
    loaded_reg = reg_prov.get_regression()
    bundle_layout = "legacy adapter" if loaded_reg.is_legacy else "canonical folder"
    scaler_name = type(loaded_reg.scaler).__name__ if loaded_reg.scaler is not None else "None (preprocessing=none)"
    print(f"✅ Regression Model đã nạp thành công: {loaded_reg.model_name}")
    print(f"   - Model Class    : {type(loaded_reg.model).__name__}")
    print(f"   - Schema Version : {loaded_reg.schema_version}")
    print(f"   - Bundle Layout  : {bundle_layout}")
    print(f"   - Scaler         : {scaler_name}")
    print(f"   - Bundle Dir     : {loaded_reg.model_dir}")
    for warning in loaded_reg.warnings:
        print(f"   - Warning        : {warning}")
except Exception as e:
    raise RuntimeError(f"❌ Nạp Regression Model thất bại: {e}") from e

print("\n🎉 Tất cả mô hình đã vượt qua Preflight! Sẵn sàng khởi động Server.")


🔍 Đang tiến hành Preflight Verification...
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
✅ YOLO Model đã nạp thành công trên thiết bị: cuda:0
📦 Đang nạp mô hình CNN từ: /content/drive/MyDrive/NGHIÊN CỨU KHOA HỌC/GROUP_MEMBERS/NGUYEN MINH TRI/MAIN_SOURCES/RESULTS/CNN_DenseNet121_Trained/best_v3_step2.keras...
✅ Nạp mô hình CNN thành công!
✅ CNN Classifier (DenseNet121) đã nạp thành công.
✅ Regression Model đã nạp thành công: ARDRegression
   - Model Class    : ARDRegression
   - Schema Version : 31v1
   - Bundle Layout  : canonical folder
   - Scaler         : StandardScaler
   - Bundle Dir     : /content/drive/MyDrive/NGHIÊN CỨU KHOA HỌC/GROUP_MEMBERS/NGUYEN MINH TRI/MAIN_SOURCES/LINEAR_REGRESSION_MODEL/models/ard

🎉 Tất cả mô hình đã vượt qua

In [5]:
# Cell 6: Foreground FastAPI Host + ngrok
import asyncio
import httpx
from pyngrok import ngrok
import uvicorn
import nest_asyncio
nest_asyncio.apply()

from rice_ai.api.application import create_app

# This cell owns the complete lifecycle. It intentionally remains running
# while the API is available; interrupting it executes the cleanup block.
app = create_app(settings, vision_provider=vision_prov, regression_provider=reg_prov)
config = uvicorn.Config(app=app, host="127.0.0.1", port=server_port, log_level="info", loop="asyncio")
server = uvicorn.Server(config)
server_task = None
current_tunnel = None
public_url = None

async def _wait_local_ready():
    for _ in range(30):
        if server_task is not None and server_task.done():
            raise RuntimeError("Uvicorn đã kết thúc trước khi readiness hoàn tất.")
        await asyncio.sleep(0.5)
        try:
            async with httpx.AsyncClient() as client:
                health = await client.get(f"http://127.0.0.1:{server_port}/health", timeout=1.0)
                status = await client.get(f"http://127.0.0.1:{server_port}/api/status", timeout=1.0)
                if health.status_code == 200 and status.status_code == 200:
                    return status.json()
        except Exception:
            pass
    raise RuntimeError("Server không phản hồi /health và /api/status sau 15 giây.")

try:
    server_task = asyncio.create_task(server.serve())
    status_payload = await _wait_local_ready()
    print(f"✅ Server nội bộ READY: {status_payload.get('readiness')}")
    if not ngrok_auth_token:
        raise ValueError("Thiếu NGROK_AUTH_TOKEN trong .env.")
    ngrok.set_auth_token(ngrok_auth_token)
    domain = ngrok_domain.replace("https://", "").replace("http://", "") if ngrok_domain else None
    current_tunnel = ngrok.connect(server_port, domain=domain) if domain else ngrok.connect(server_port)
    public_url = current_tunnel.public_url
    print("=" * 62)
    print(f"🚀 NGROK PUBLIC URL: {public_url}")
    print("Cell này sẽ tiếp tục chạy. Dừng bằng Interrupt/Stop để đóng server và tunnel.")
    print("=" * 62)
    if ENV_FILE.is_file():
        lines = ENV_FILE.read_text(encoding="utf-8").splitlines(keepends=True)
        replaced = False
        updated = []
        for line in lines:
            if line.strip().startswith("AI_SERVER_URL="):
                updated.append(f"AI_SERVER_URL={public_url}\n")
                replaced = True
            else:
                updated.append(line)
        if not replaced:
            updated.append(f"AI_SERVER_URL={public_url}\n")
        ENV_FILE.write_text("".join(updated), encoding="utf-8")
    await server_task
finally:
    if current_tunnel is not None:
        try:
            ngrok.disconnect(current_tunnel.public_url)
            print(f"✅ Đã đóng ngrok tunnel: {current_tunnel.public_url}")
        except Exception as exc:
            print(f"⚠️ Không thể đóng tunnel: {exc}")
        current_tunnel = None
    if server is not None:
        server.should_exit = True
    if server_task is not None and not server_task.done():
        try:
            await asyncio.wait_for(server_task, timeout=10.0)
        except asyncio.TimeoutError:
            print("⚠️ Uvicorn chưa dừng trong 10 giây.")
    print("🎉 Host cell đã kết thúc an toàn.")


INFO:     Started server process [3368]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:38680 - "GET /health HTTP/1.1" 200 OK
INFO:     127.0.0.1:38680 - "GET /api/status HTTP/1.1" 200 OK
✅ Server nội bộ READY: ready
🚀 NGROK PUBLIC URL: https://provolone-duress-probably.ngrok-free.dev
Cell này sẽ tiếp tục chạy. Dừng bằng Interrupt/Stop để đóng server và tunnel.
Performing prediction on 54 slices.
INFO:     118.69.159.154:0 - "POST /predict HTTP/1.1" 200 OK
Performing prediction on 54 slices.
INFO:     113.182.78.152:0 - "POST /predict HTTP/1.1" 200 OK


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [3368]


✅ Đã đóng ngrok tunnel: https://provolone-duress-probably.ngrok-free.dev
🎉 Host cell đã kết thúc an toàn.


In [6]:
# Cell 7: Smoke test
# Host cell 6 phải đang chạy. Thực hiện smoke/E2E từ Node.js, React hoặc một client bên ngoài.
# Không chạy cell này để khởi động server nền.
pass


In [7]:
# Cell 8: Sau khi interrupt Cell 6
print("Cell 6 đã tự cleanup server và ngrok trong finally.")
pass


Cell 6 đã tự cleanup server và ngrok trong finally.
